## 持久化

#### 运行时上下文

In [ ]:
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from dataclasses import dataclass
from langgraph.runtime import Runtime
from langgraph.graph import StateGraph, START, END
from typing import Literal, TypedDict


@dataclass
class UserContext:
    username: str
    level: Literal["standard", "vip"]

class State(TypedDict):
    user_input: str
    output: str

support_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """你是售后客服助手。
客户称呼：{username}
客户等级：{level}

根据客户等级调整称呼和解释详略，但保持礼貌、准确。
不要仅凭客户等级承诺退款、补偿或其他权益；遇到这类问题，
说明需要依据订单信息和业务规则核实。"""
    ),
    ("human", "{user_input}"),
])

def llm_node(state: State, runtime: Runtime[UserContext]):
    prompt_value = support_prompt.invoke({
        "username": runtime.context.username,
        "level": runtime.context.level,
        "user_input": state["user_input"],
    })

    ai_msg = model.invoke(prompt_value)
    return {"output": ai_msg.content}


builder = StateGraph(State, context_schema=UserContext)
builder.add_node(llm_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", END)
graph = builder.compile()

result = graph.invoke(
    {"user_input": "我的订单晚到了，怎么处理？"},
    context=UserContext(username="Alice", level="vip")
)
print(result["output"])